# AgentMorph Stage 1 — AgentDojo canary

Runs `Llama-3.2-3B` × `native` × **AgentDojo** (workspace suite) × 5 scenarios.

This notebook ticks the **"one AgentDojo sweep on the canary model"** box from the 16-day realistic plan. It's a minimal smoke test:

- Only the native framework (proven on e-commerce; LangGraph on AgentDojo deferred).
- Only 5 tasks (`max_tasks=5`) to keep wall time < 15 min.
- Workspace suite — smallest of the four AgentDojo suites.

**Goal:** produce at least 3 clean trajectories from AgentDojo and confirm the adapter handles whatever AgentDojo version you have installed. If it works, add `--env agentdojo` to the regular per-model notebooks for a wider sweep later.

## 1. GPU check

In [ ]:
# Soft GPU check — cells 2-6 (adapter probe, suite discovery) run on CPU.
# Only §7 (the actual sweep) needs a T4.
import torch
if torch.cuda.is_available():
    print('CUDA:', torch.cuda.get_device_name(0),
          f'({torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB)')
    _HAS_GPU = True
else:
    print('NOTE: no CUDA visible. Cells 2-6 still work on CPU. '
          'Switch to a T4 GPU runtime before running §7 (sweep).')
    _HAS_GPU = False

## 2. Clone / pull the repo

In [ ]:
import os
REPO_URL = "https://github.com/Pranamya16/AgentMorph.git"
REPO_DIR = "/content/AgentMorph"
if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !cd {REPO_DIR} && git fetch origin && git checkout main && git pull --ff-only
!cd {REPO_DIR} && git log -1 --oneline

## 3. HF auth

In [ ]:
from huggingface_hub import login
HF_TOKEN = "hf_REPLACE_ME"
login(token=HF_TOKEN)

## 4. Mount Drive + install (includes AgentDojo extra)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install -q -e /content/AgentMorph[models,langgraph,agentdojo]

## 5. Verify the AgentDojo adapter sees suites

Runs a lightweight probe (no GPU needed) to confirm the version of AgentDojo you installed exposes an API shape the adapter recognises. If this fails, the full sweep below will also fail, and you should paste the error back for a patch.

In [ ]:
from agentmorph.environments.agentdojo_env import AgentDojoEnvironment
env = AgentDojoEnvironment(suite="workspace", max_tasks=5)
print("available suites:", env.available_suites())
print("picked suite:", env.suite)
# Count user tasks in the picked suite.
scenarios = list(env.scenarios())
print(f"scenarios exposed: {len(scenarios)}")
if scenarios:
    s = scenarios[0]
    print("first scenario id:", s.id)
    print("first scenario prompt (first 200 chars):", s.prompt[:200])

## 6. (Optional) wipe a prior AgentDojo run

Only needed if a previous run for this model left bad trajectories on AgentDojo cells.

In [ ]:
import json, pathlib
MODEL = "Llama-3.2-3B"
RUN_DIR = pathlib.Path("/content/drive/MyDrive/AgentMorph/runs/stage1_agentdojo")
manifest_path = RUN_DIR / "manifest.json"
traj_dir = RUN_DIR / "trajectories"

if manifest_path.exists():
    data = json.loads(manifest_path.read_text())
    before = len(data.get("completed", {}))
    data["completed"] = {k: v for k, v in data.get("completed", {}).items() if not k.startswith(MODEL + "|")}
    after = len(data["completed"])
    manifest_path.write_text(json.dumps(data, indent=2))
    print(f"manifest: {before} -> {after} entries")
else:
    print("no manifest - nothing to prune")

if traj_dir.exists():
    for p in traj_dir.glob(f"{MODEL}__*.jsonl"):
        p.unlink()
        print("deleted:", p.name)

## 7. Sweep — `Llama-3.2-3B` × native × AgentDojo × 5 tasks

Expect wall time ~5-10 min: model load (~2 min from Drive cache) + 5 tasks × ~30-60 s each.

In [ ]:
import os
# §7 runs the real sweep — Llama-3.2-3B at 4-bit needs a T4.
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
!PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True python -m agentmorph.runner \
    --model Llama-3.2-3B --framework native --env agentdojo --n-scenarios 5 \
    --hf-cache-dir /content/drive/MyDrive/AgentMorph/hf_cache \
    --out-dir /content/drive/MyDrive/AgentMorph/runs/stage1_agentdojo

## 8. Inspect results

In [ ]:
import json, pathlib
traj_dir = pathlib.Path("/content/drive/MyDrive/AgentMorph/runs/stage1_agentdojo/trajectories")
for p in sorted(traj_dir.glob("*.jsonl")):
    print(f"\n=== {p.name} ===")
    total = 0
    finished = 0
    for line in p.open():
        t = json.loads(line)
        total += 1
        if t.get("final_answer"):
            finished += 1
        marker = "OK " if t.get("final_answer") else "ERR"
        steps = len(t.get("steps", []))
        print(f"  [{marker}] {t['scenario_id']:50s} steps={steps:2d}")
    pct = 100 * finished / total if total else 0
    print(f"  -> finished {finished}/{total} ({pct:.0f}%)")

## 9. If something fails

Paste the FULL error text from the first error step back into Claude:

```python
import json, pathlib
p = next(pathlib.Path('/content/drive/MyDrive/AgentMorph/runs/stage1_agentdojo/trajectories').glob('*.jsonl'))
for line in p.open():
    t = json.loads(line)
    errs = [s for s in t['steps'] if s['kind'] == 'error']
    if errs:
        print(errs[0]['content'])
        break
```

Most likely failure modes:
- `ImportError: AgentDojo is installed but no recognized suite API was found.` -> paste your `agentdojo` version (`pip show agentdojo | grep Version`) back for a patch.
- A missing AgentDojo tool signature field -> adapter falls back to string type; some scenarios may not run, but others should.